# 8.2 Peer-Review — 가상 발표를 리뷰어로 재실행

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SuminHan/book-ml/blob/main/notebooks/ml2/chapter08_2_peer_review_sim.ipynb)

책 본문: [8.2절](https://smhanlab.com/book-ml/kor/ml2/chapter08/2.html)

본문 §"손으로 한 번"의 **가상 발표**를 리뷰어의 눈으로 재실행합니다. 발표팀의 실험(CliffWalking, SARSA vs Q-learning, 시드 0, $\alpha=0.5$, $\gamma=1.0$, $\varepsilon=0.1$ 고정, 500 에피소드)을 그대로 재현한 뒤, 발표가 빠뜨린 세 가지 — **평가 $\varepsilon$**, **시드 2개 추가**, **베이스라인(무작위 정책)** — 을 차례로 채워보며 §"5가지 패턴"의 1·2·5번에 대한 "반박 가능한 질문"에 실제로 답하는 실험을 합니다.

## 1. 발표팀 실험 재현: 시드 0, $\varepsilon=0.1$ 고정, *평가도* $\varepsilon=0.1$

6.3절 노트북의 환경·알고리즘 코드를 그대로 재사용합니다. 발표팀이 "최종 평가(학습된 정책으로 200 에피소드)"라고만 쓰고 $\varepsilon$을 명시하지 않은 부분을 **평가도 $\varepsilon=0.1$**으로 해석해서 재현합니다 — 이것이 발표에서 나온 "SARSA −21.3, Q-learning −45.3"의 숫자입니다.

In [1]:
import gymnasium as gym
import random

env = gym.make("CliffWalking-v1")
n_states, n_actions = env.observation_space.n, env.action_space.n
rows, cols = env.unwrapped.shape
cliff = {r*cols+c for r in range(rows) for c in range(cols) if env.unwrapped._cliff[r][c]}
print(f"격자 {rows}×{cols}, 상태 {n_states}개, 행동 {n_actions}개, 절벽 {len(cliff)}개")

def epsilon_greedy(Q, s, epsilon, n_actions):
    if random.random() < epsilon:
        return random.randrange(n_actions)
    return max(range(n_actions), key=lambda a: Q[s][a])

def train(mode, n_episodes=500, alpha=0.5, gamma=1.0, epsilon=0.1, seed=0):
    random.seed(seed)
    Q = [[0.0]*n_actions for _ in range(n_states)]
    returns = []
    for ep in range(n_episodes):
        s, _ = env.reset(seed=ep)
        a = epsilon_greedy(Q, s, epsilon, n_actions)
        tot, steps = 0.0, 0
        for _ in range(500):
            ns, r, term, trunc, _ = env.step(a)
            tot += r; steps += 1
            done = term or trunc
            na = epsilon_greedy(Q, ns, epsilon, n_actions)
            if mode == "q":
                tgt = r + (gamma*max(Q[ns]) if not done else 0.0)   # max (off-policy)
            else:
                tgt = r + (gamma*Q[ns][na] if not done else 0.0)   # 실제로 고른 na (on-policy)
            Q[s][a] += alpha*(tgt - Q[s][a])
            s, a = ns, na
            if done:
                break
        returns.append((tot, steps))
    return Q, returns

def evaluate(Q, n_ep=200, seed=0, epsilon=0.0):
    """학습된 Q를 epsilon-greedy(epsilon=0이면 탐욕적)로 n_ep 에피소드 평가."""
    random.seed(seed)
    rets, steps_list = [], []
    for ep in range(n_ep):
        s, _ = env.reset(seed=ep); tot, steps = 0.0, 0
        for _ in range(500):
            a = epsilon_greedy(Q, s, epsilon, n_actions)
            ns, r, term, trunc, _ = env.step(a); tot += r; steps += 1
            s = ns
            if term or trunc: break
        rets.append(tot); steps_list.append(steps)
    return sum(rets)/n_ep, sum(steps_list)/n_ep

Q_sarsa, rets_sarsa = train("sarsa")
Q_q,     rets_q     = train("q")
print("학습 완료: 시드 0, alpha=0.5, gamma=1.0, epsilon=0.1 고정, 500 에피소드")

격자 4×12, 상태 48개, 행동 4개, 절벽 10개
학습 완료: 시드 0, alpha=0.5, gamma=1.0, epsilon=0.1 고정, 500 에피소드


In [2]:
# 발표팀의 "최종 평가" 재현: 평가도 epsilon=0.1로 (발표에서 epsilon 미명시)
s21, s21_st = evaluate(Q_sarsa, epsilon=0.1)
q45, q45_st = evaluate(Q_q, epsilon=0.1)
print("발표팀 재현 (평가 epsilon=0.1, 200 에피소드):")
print(f"  SARSA:      {s21:.1f}  (평균 {s21_st:.0f} 스텝)")
print(f"  Q-learning: {q45:.1f}  (평균 {q45_st:.0f} 스텝)")
print()
print("-> 발표의 결론: 'SARSA(-21) > Q-learning(-45)' — SARSA가 더 좋은 알고리즘")

발표팀 재현 (평가 epsilon=0.1, 200 에피소드):
  SARSA:      -21.3  (평균 19 스텝)
  Q-learning: -45.3  (평균 17 스텝)

-> 발표의 결론: 'SARSA(-21) > Q-learning(-45)' — SARSA가 더 좋은 알고리즘


## 2. 패턴 2 검증 — 평가 $\varepsilon$을 0으로 (리뷰 질문 재실행)

리뷰어의 3점 질문: *"최종 평가는 $\varepsilon=0$이었나요?"* — **학습 결과는 그대로 둔 채** 평가만 $\varepsilon=0$(탐욕적)으로 바꿉니다. 본문 §"실습" §2의 손계산(절벽과 한 스텝인 Q-learning 경로: 추락 확률 1차, 두 스텝인 SARSA 우회 경로: 2차)이 시뮬레이션과 맞는지 확인합니다.

In [3]:
# 같은 Q 테이블을 epsilon=0(탐욕적)으로 재평가
s_g, s_g_st = evaluate(Q_sarsa, epsilon=0.0)
q_g, q_g_st = evaluate(Q_q, epsilon=0.0)
print("같은 학습 결과, 평가 epsilon=0 (탐욕적, 200 에피소드):")
print(f"  SARSA:      {s_g:.1f}  (평균 {s_g_st:.0f} 스텝)")
print(f"  Q-learning: {q_g:.1f}  (평균 {q_g_st:.0f} 스텝)")
print()
print("손계산 대조: Q-learning ~ -38(절벽 한 스텝, 1차 추락), SARSA ~ -20(절벽 두 스텝, 2차 추락)")
print(f"실측:        Q-learning {q45:.1f} (eps=0.1) -> {q_g:.1f} (eps=0),  SARSA {s21:.1f} (eps=0.1) -> {s_g:.1f} (eps=0)")
print()
print("-> 평가 epsilon만 0.1 -> 0으로 바꾸자 Q-learning -45 -> -13으로 급상승하고")
print("   SARSA는 -21 -> -17로만 올라감. epsilon=0 기준으로는 Q-learning이 더 낫다.")
print("   즉 발표의 'SARSA가 더 좋다'는 결론은 평가 프로토콜의 선택에 좌우된다.")

같은 학습 결과, 평가 epsilon=0 (탐욕적, 200 에피소드):
  SARSA:      -17.0  (평균 17 스텝)
  Q-learning: -13.0  (평균 13 스텝)

손계산 대조: Q-learning ~ -38(절벽 한 스텝, 1차 추락), SARSA ~ -20(절벽 두 스텝, 2차 추락)
실측:        Q-learning -45.3 (eps=0.1) -> -13.0 (eps=0),  SARSA -21.3 (eps=0.1) -> -17.0 (eps=0)

-> 평가 epsilon만 0.1 -> 0으로 바꾸자 Q-learning -45 -> -13으로 급상승하고
   SARSA는 -21 -> -17로만 올라감. epsilon=0 기준으로는 Q-learning이 더 낫다.
   즉 발표의 'SARSA가 더 좋다'는 결론은 평가 프로토콜의 선택에 좌우된다.


In [4]:
# 평가 epsilon을 0 ~ 0.3까지 스윕: 두 알고리즘의 "평가 리턴 곡선"
import numpy as np
eps_grid = np.arange(0.0, 0.31, 0.01)
q_curve = [evaluate(Q_q, epsilon=e)[0] for e in eps_grid]
s_curve = [evaluate(Q_sarsa, epsilon=e)[0] for e in eps_grid]

import matplotlib
matplotlib.use("Agg")
from matplotlib import font_manager
import matplotlib.pyplot as plt
kr = [f.name for f in font_manager.fontManager.ttflist if "Noto Sans CJK KR" in f.name]
if kr: plt.rcParams["font.sans-serif"] = [kr[0]]
plt.rcParams["axes.unicode_minus"] = False

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(eps_grid, s_curve, color="#4a90d9", lw=2, label=f"SARSA (17스텝 우회)")
ax.plot(eps_grid, q_curve, color="#d9534f", lw=2, label=f"Q-learning (13스텝 절벽 옆)")
ax.axvline(0.0, color="gray", ls=":", lw=1)
ax.axvline(0.1, color="gray", ls="--", lw=1)
ax.text(0.003, ax.get_ylim()[1]-1, r"$\varepsilon=0$ (greedy)", fontsize=8, color="gray")
ax.text(0.103, ax.get_ylim()[1]-1, r"$\varepsilon=0.1$ (발표팀 설정)", fontsize=8, color="gray")
ax.set_xlabel(r"평가 $\varepsilon$")
ax.set_ylabel("평가 평균 리턴 (200 에피소드)")
ax.set_title(r"평가 $\varepsilon$의 비대칭: 탐색(ε>0)이 조금만 섞이면 Q-learning이 급격히 떨어져 순위가 뒤집힌다")
ax.legend(loc="lower left", fontsize=9)
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig("ch08_2_eval_epsilon_asymmetry_local.svg", bbox_inches="tight")
plt.show()

# 평가 epsilon이 오를수록 Q-learning만 급격히 떨어짐을 요약 출력
for e, qs, qq in zip(eps_grid, s_curve, q_curve):
    if abs(e - 0.0) < 1e-9 or abs(e - 0.05) < 1e-9 or abs(e - 0.1) < 1e-9 or abs(e - 0.3) < 1e-9:
        lead = "SARSA" if qs > qq else "Q-learning"
        print(f"  epsilon={e:.2f}: SARSA {qs:6.1f}, Q-learning {qq:6.1f}  -> {lead}가 앞섬")
print()
print("-> epsilon=0(탐욕적)에서만 Q-learning이 앞서고, 탐색이 조금만 섞이면(SARSA 우위)")
print("   Q-learning의 리턴만 급격히 떨어짐. '평가 epsilon'이 결론의 방향을 가르는 변수.")
print("그림 저장: ch08_2_eval_epsilon_asymmetry_local.svg")

  epsilon=0.00: SARSA  -17.0, Q-learning  -13.0  -> Q-learning가 앞섬
  epsilon=0.05: SARSA  -20.5, Q-learning  -29.7  -> SARSA가 앞섬
  epsilon=0.10: SARSA  -21.3, Q-learning  -45.3  -> SARSA가 앞섬
  epsilon=0.30: SARSA  -39.0, Q-learning -181.6  -> SARSA가 앞섬

-> epsilon=0(탐욕적)에서만 Q-learning이 앞서고, 탐색이 조금만 섞이면(SARSA 우위)
   Q-learning의 리턴만 급격히 떨어짐. '평가 epsilon'이 결론의 방향을 가르는 변수.
그림 저장: ch08_2_eval_epsilon_asymmetry_local.svg


## 3. 패턴 1 검증 — 시드 1, 2 추가

리뷰어의 두 번째 질문: *"시드를 2개 더 돌리면 같은 모양이 나올까요?"* — 발표팀이 시드 0 하나로 결론 낸 것을, 8.1 §"비교 프로토콜"의 시드 3개 프로토콜로 재실행합니다. greedy 평가(\$\varepsilon=0\$)를 시드별로 표로 뽑습니다.

In [5]:
print("시드별 greedy(epsilon=0) 평가 — 8.1 §비교 프로토콜:")
print()
print("| 시드 | SARSA greedy | Q-learning greedy |")
print("|---|---|---|")
for seed in (0, 1, 2):
    if seed == 0:
        s_val, _ = evaluate(Q_sarsa, epsilon=0.0)
        q_val, _ = evaluate(Q_q, epsilon=0.0)
    else:
        Q_s, _ = train("sarsa", seed=seed)
        Q_q2, _ = train("q", seed=seed)
        s_val, _ = evaluate(Q_s, epsilon=0.0)
        q_val, _ = evaluate(Q_q2, epsilon=0.0)
    print(f"| {seed} | {s_val:.1f} | {q_val:.1f} |")
print()
print("-> 이 설정(Gymnasium CliffWalking-v1, alpha=0.5, eps=0.1, 500에피소드)에서는")
print("   시드 3개 모두 Q-learning -13, SARSA -17로 일관된다. 단, 8.1의 표처럼 설정이")
print("   달라지면 SARSA 시드 중 하나가 -200으로 고착되는 경우도 있다 — '시드 3개'가")
print("   형식이 아니라 정량적 요구인 이유. '시드 1개'로는 이 일관성 자체를 확인할 수 없다.")

시드별 greedy(epsilon=0) 평가 — 8.1 §비교 프로토콜:

| 시드 | SARSA greedy | Q-learning greedy |
|---|---|---|
| 0 | -17.0 | -13.0 |
| 1 | -17.0 | -13.0 |


| 2 | -17.0 | -13.0 |

-> 이 설정(Gymnasium CliffWalking-v1, alpha=0.5, eps=0.1, 500에피소드)에서는
   시드 3개 모두 Q-learning -13, SARSA -17로 일관된다. 단, 8.1의 표처럼 설정이
   달라지면 SARSA 시드 중 하나가 -200으로 고착되는 경우도 있다 — '시드 3개'가
   형식이 아니라 정량적 요구인 이유. '시드 1개'로는 이 일관성 자체를 확인할 수 없다.


## 4. 패턴 5 검증 — 베이스라인(무작위 정책) 측정

리뷰어의 세 번째 질문: *"무작위 정책의 평균 리턴은 얼마인가요?"* — "잘 배웠다"는 *측정된* 기준점(무작위 정책 리턴) 위에서야 의미 있다. 8.1의 M2 마일스톤이 베이스라인 숫자 한 줄을 제출물로 요구하는 이유를 숫자로 확인합니다.

In [6]:
random.seed(0)
tot_all, steps_all = [], []
for ep in range(500):
    s, _ = env.reset(seed=ep); tot, steps = 0.0, 0
    for _ in range(500):
        a = random.randrange(n_actions)
        ns, r, term, trunc, _ = env.step(a); tot += r; steps += 1
        s = ns
        if term or trunc: break
    tot_all.append(tot); steps_all.append(steps)
print(f"무작위 정책 (500 에피소드): 평균 리턴 {sum(tot_all)/500:.0f}, 평균 스텝 {sum(steps_all)/500:.0f}")
print(f"배운 정책 greedy: SARSA {s_g:.1f}, Q-learning {q_g:.1f}")
print(f"-> 무작위 대비 이득: SARSA {s_g - sum(tot_all)/500:.0f}, Q-learning {q_g - sum(tot_all)/500:.0f}")
print("   '잘 배웠다'는 이 측정된 기준점 위에서야 검증 가능한 주장이다.")

무작위 정책 (500 에피소드): 평균 리턴 -5036, 평균 스텝 486
배운 정책 greedy: SARSA -17.0, Q-learning -13.0
-> 무작위 대비 이득: SARSA 5019, Q-learning 5023
   '잘 배웠다'는 이 측정된 기준점 위에서야 검증 가능한 주장이다.


## 5. 두 greedy 경로를 격자에 그리기

왜 평가 $\varepsilon$의 효과가 비대칭인지를 구조적으로 봅니다 — Q-learning의 경로는 **절벽 바로 위(2행)**, SARSA의 경로는 **위쪽(0~1행) 우회**입니다(6.3절의 경로 그림과 동일한 구조, 8.2에서는 평가 $\varepsilon$의 관점에서 다시 읽습니다).

In [7]:
def greedy_path(Q, seed=0):
    random.seed(seed)
    s, _ = env.reset(seed=0); path = [s]
    for _ in range(500):
        a = max(range(n_actions), key=lambda a: Q[s][a])
        ns, r, term, trunc, _ = env.step(a); s = ns; path.append(s)
        if term or trunc: break
    return path

path_sarsa = greedy_path(Q_sarsa)
path_q = greedy_path(Q_q)
print(f"SARSA 탐욕적 경로: {len(path_sarsa)-1} 스텝, 사용 행 = {sorted(set(x//cols for x in path_sarsa))}")
print(f"Q-learning 탐욕적 경로: {len(path_q)-1} 스텝, 사용 행 = {sorted(set(x//cols for x in path_q))}")

def draw_cell(ax, s, color, edgecolor='none', zorder=2, alpha=1.0):
    r, c = s//cols, s%cols
    ax.add_patch(plt.Rectangle((c, rows-1-r), 1, 1, facecolor=color, edgecolor=edgecolor, zorder=zorder, alpha=alpha))

fig, ax = plt.subplots(figsize=(10, 4))
for s in range(n_states):
    col = "#1a1a1a" if s in cliff else "white"
    draw_cell(ax, s, col, edgecolor='gray', zorder=1)
for s in path_sarsa[:-1]:
    if s not in cliff: draw_cell(ax, s, "#4a90d9", alpha=0.7, zorder=2)
for s in path_q[:-1]:
    if s not in cliff: draw_cell(ax, s, "#d9534f", alpha=0.9, zorder=3)
draw_cell(ax, path_sarsa[0], "white", edgecolor='black', zorder=4)
draw_cell(ax, path_sarsa[-1], "white", edgecolor='black', zorder=4)
ax.text(0.35, rows-0.35, "S", ha='center', va='center', fontsize=14, color='black')
ax.text(cols-0.35, rows-0.35, "G", ha='center', va='center', fontsize=14, color='black')
ax.set_xlim(-0.1, cols+0.1); ax.set_ylim(-0.1, rows+0.1)
ax.set_aspect("equal"); ax.axis("off")
ax.set_title(r"CliffWalking: Q-learning(빨강, 절벽 바로 위) — 평가 $\varepsilon$에 민감 / SARSA(파랑, 위쪽 우회) — 덜 민감")
from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(facecolor="#d9534f", label="Q-learning (절벽과 한 스텝)"),
    Patch(facecolor="#4a90d9", alpha=0.7, label="SARSA (절벽과 두 스텝)"),
    Patch(facecolor="#1a1a1a", label="절벽 (-100)"),
], loc='upper right', fontsize=9)
fig.tight_layout()
plt.show()
print("경로 구조가 평가 epsilon 비대칭의 기원: 한 스텝(1차 추락) vs 두 스텝(2차 추락)")

SARSA 탐욕적 경로: 17 스텝, 사용 행 = [0, 1, 2, 3]
Q-learning 탐욕적 경로: 13 스텝, 사용 행 = [2, 3]
경로 구조가 평가 epsilon 비대칭의 기원: 한 스텝(1차 추락) vs 두 스텝(2차 추락)


## 정리: 가상 발표에 3점 리뷰를 쓰자

실습에서 나온 숫자를 근거로, 본문 §"손으로 한 번"의 가상 발표를 향해 3점 리뷰를 써봅니다.

```
1. 환경 설명 — CliffWalking(4×12, 절벽 -100) 명시. 주 지표가 '최종 평가 리턴'인지
   '학습중 리턴'인지, 평가 epsilon이 0인지가 보고서에 없다.
2. 알고리즘 선택 — SARSA vs Q-learning은 절벽 -100의 보상 구조에 맞는 비교(통과).
3. 수식적 정당화 — Ch06.3을 인용했지만 *관측 숫자*가 없다. 'SARSA가 안전 알고리즘이라'
   는 설명은 학습중 성과에 관한 것이지 최종 평가에 관한 것이 아니다.
4. 결과의 정직성 — 시드 1개, 베이스라인 없음, 평가 epsilon 미명시.
5. 반박 가능한 질문 —
   근거 개념: Ch06.3 (on/off-policy 경로 차이)
   답에 필요한 새 실험: epsilon=0으로 200에피소드 재평가 + 시드 1, 2 추가 + 무작위 베이스라인
   질문: "최종 평가는 epsilon=0이었나요? epsilon=0.1로 평가하면 SARSA -21 vs Q-learning
         -45로 'SARSA가 좋다'가 되지만, epsilon=0으로 재평가하면 Q-learning -13 vs
         SARSA -17으로 순위가 뒤집힙니다(실습 §2). 두 프로토콜 모두 시드 3개로
         재실행해 주세요. 무작위 정책 리턴(-5000) 대비 이득도 같이 보고해 주세요."
```

3단계(§채점 기준) 중 3점에 도달하려면 5번의 두 칸 — **근거 개념**과 **답에 필요한 새 실험** — 이 모두 채워져야 한다. 이번 실습이 바로 그 "새 실험"의 내용이다: 리뷰어는 발표팀의 실험을 *자기 손으로* 재실행할 수 있어야, 날카로운 질문을 던질 수 있다.